# W88 — HumanEval Sequential-Reflexion Bench (Colab path)

Cross-model robustness check for the W88 HumanEval head-to-head.  The
primary W88 run is on NIM Llama-3.1-8B-Instruct (directly comparable
to the W86 negative result this milestone is built to retire).  This
Colab notebook reproduces the same bench against a stronger
open-weight code-LM (Qwen2.5-Coder-7B-Instruct) at Colab Pro
A100-40GB scale.

**Retirement bars** (`docs/RUNBOOK_W88.md`):
1. `b_mean_strictly_beats_a1_mean = True`
2. `b_mean − a1_mean ≥ +1.0 pp`
3. `b_mean_strictly_beats_a0_mean = True`
4. B beats A1 on more than half the seeds.

## One-time setup

1. **Runtime → Change runtime type → A100 GPU** (L4 works too;
   T4 is borderline).
2. **HF Secret** (optional, only if Qwen2.5-Coder-7B needs auth):
   key icon → `hf_token` secret.  Qwen models are public so this
   is usually unnecessary; the cell auto-skips if absent.
3. *Runtime → Run all*.  Wall ~30-45 min on A100 for 30 problems ×
   3 seeds × 3 arms ≈ 990 model calls.  Drive copy at the end.

In [ ]:
# --- 1. Environment probe + workdir ---
import os, sys, subprocess, json, datetime
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = f'/content/w88_humaneval_reflexion_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('RUN_TS:', RUN_TS)
print('OUT_DIR:', OUT_DIR)
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(
        f'GPU: {p.name}  mem={p.total_memory/(1024**3):.2f} GiB  '
        f'sm={p.major}.{p.minor}')

In [ ]:
# --- 2. Install transformers + accelerate ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0', 'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0', 'numpy>=1.24',
], check=True)
import transformers, accelerate
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)

In [ ]:
# --- 3. (Optional) HF login ---
try:
    from google.colab import userdata  # type: ignore
    hf_token = (userdata.get('hf_token') or '').strip()
    if hf_token.startswith('hf_'):
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token, add_to_git_credential=False)
        print('HF login OK (token len =', len(hf_token), ')')
    else:
        print('No hf_token secret; using anonymous HF access (fine for Qwen).')
except Exception as e:  # noqa: BLE001
    print('HF login skipped:', type(e).__name__, e)

In [ ]:
# --- 4. Clone CoordPy ---
import shutil
shutil.rmtree('/content/CoordPy', ignore_errors=True)
subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/adotdong29/CoordPy.git',
    '/content/CoordPy',
], check=True)
os.chdir('/content/CoordPy')
print('HEAD =', subprocess.check_output(
    ['git', 'log', '-1', '--oneline'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__,
      'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 5. Load Qwen2.5-Coder-7B-Instruct ---
from transformers import (
    AutoModelForCausalLM, AutoTokenizer)
import torch, time
MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
print('loading', MODEL_ID, '...')
tok = AutoTokenizer.from_pretrained(MODEL_ID)
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16,
    device_map='auto')
model.eval()
print(f'model loaded in {time.time()-t0:.1f}s')
print('dtype:', next(model.parameters()).dtype,
      'device:', next(model.parameters()).device)

In [ ]:
# --- 6. Build a `gen(prompt, max_tokens, temperature)` ---
import torch, time
@torch.inference_mode()
def gen(prompt, max_tokens, temperature):
    msgs = [{'role': 'user', 'content': prompt}]
    input_text = tok.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(input_text, return_tensors='pt').to(
        model.device)
    t0 = time.time()
    do_sample = float(temperature) > 0.0
    gen_kwargs = dict(
        max_new_tokens=int(max_tokens),
        do_sample=do_sample,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    if do_sample:
        gen_kwargs['temperature'] = float(temperature)
        gen_kwargs['top_p'] = 0.95
    out = model.generate(**inputs, **gen_kwargs)
    wall = int((time.time() - t0) * 1000)
    response_ids = out[0, inputs['input_ids'].shape[1]:]
    text = tok.decode(response_ids, skip_special_tokens=True)
    return text, wall
print('warm-up...')
_warm, _ = gen('Reply with exactly the word OK.', 16, 0.0)
print('warm-up response:', repr(_warm)[:120])

In [ ]:
# --- 7. Run the W88 HumanEval reflexion bench ---
from coordpy.humaneval_real_bench_v1 import (
    load_humaneval_corpus_v1)
from coordpy.humaneval_reflexion_bench_v1 import (
    HumanEvalReflexionBenchConfigV1,
    run_humaneval_reflexion_bench_v1)
import json, time, hashlib
corpus = load_humaneval_corpus_v1()
print('corpus:', len(corpus), 'problems')
N_PROBLEMS = 30
N_SEEDS = 3
seeds = tuple(88_028_001 + i for i in range(N_SEEDS))
cfg = HumanEvalReflexionBenchConfigV1(
    n_problems=N_PROBLEMS, K_multi_sample=5,
    seeds=seeds, sampling_temperature=0.7,
    max_tokens_per_call=768)
sidecar_path = OUT_DIR + '/humaneval_reflexion_calls.jsonl'
sidecar_f = open(sidecar_path, 'w')
n_calls = 0
def wrapped_gen(prompt, max_tokens, temperature):
    global n_calls
    n_calls += 1
    text, wall = gen(prompt, max_tokens, temperature)
    rec = {
        'model_id': MODEL_ID, 'backend': 'colab_hf_local',
        'n_call': int(n_calls),
        'prompt_sha256': hashlib.sha256(
            prompt.encode('utf-8')).hexdigest(),
        'response_sha256': hashlib.sha256(
            text.encode('utf-8')).hexdigest(),
        'temperature': float(temperature),
        'max_tokens': int(max_tokens),
        'wall_ms': int(wall),
        'prompt': prompt, 'response_text': text}
    sidecar_f.write(json.dumps(rec, separators=(',', ':')) + '\n')
    sidecar_f.flush()
    return text, int(wall)
t0 = time.time()
def progress(seed, p_idx, task_id):
    elapsed = time.time() - t0
    print(
        f'  seed={seed} problem {p_idx+1}/{N_PROBLEMS} '
        f'(task={task_id}) elapsed={elapsed:.0f}s '
        f'calls={n_calls}', flush=True)
report = run_humaneval_reflexion_bench_v1(
    gen=wrapped_gen, model_id=MODEL_ID, corpus=corpus,
    config=cfg, on_problem_start=progress)
sidecar_f.close()
dt = time.time() - t0
print()
print(f'wall: {dt:.0f}s, calls: {n_calls}')
print(f'A0 mean pass@1: {report.a0_mean_pass_at_1:.4f}')
print(f'A1 mean pass@1: {report.a1_mean_pass_at_1:.4f}')
print(f'B  mean pass@1: {report.b_mean_pass_at_1:.4f}')
print(
    f'B − A1: {report.b_mean_minus_a1_mean_pp:+.2f} pp')
print(
    f'B mean strictly > A1 mean: '
    f'{report.b_mean_strictly_beats_a1_mean}')
report_path = (
    OUT_DIR + '/humaneval_reflexion_bench_report.json')
with open(report_path, 'w') as f:
    json.dump(report.to_dict(), f, indent=2)
print('report ->', report_path)

In [ ]:
# --- 8. Offline re-verifier ---
!cd /content/CoordPy && python scripts/verify_w88_humaneval_reflexion_audit_chain.py --run-dir {OUT_DIR}

In [ ]:
# --- 9. Save to Drive + download zip ---
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    DEST = (
        '/content/drive/MyDrive/coordpy_w88_humaneval_reflexion/'
        f'w88_{RUN_TS}')
    os.makedirs(DEST, exist_ok=True)
    subprocess.run(
        f'cp -r {OUT_DIR}/. {DEST}/', shell=True, check=False)
    print('saved to Drive:', DEST)
    subprocess.run(
        f'ls -la {DEST}', shell=True, check=False)
except Exception as e:  # noqa: BLE001
    print('Drive copy skipped:', type(e).__name__, e)
zip_path = f'/content/w88_humaneval_reflexion_{RUN_TS}.zip'
subprocess.run([
    'zip', '-rq', zip_path,
    f'w88_humaneval_reflexion_{RUN_TS}',
], cwd='/content', check=False)
print('zip:', zip_path,
      f'({os.path.getsize(zip_path)/1024:.1f} KiB)')
try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except Exception as e:  # noqa: BLE001
    print('files.download skipped:', e)